In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import string
import os
from google.colab import files

# Upload the dataset to the Colab environment
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"Successfully uploaded: {file_name}")

Saving next_word_predictor.txt to next_word_predictor.txt
Successfully uploaded: next_word_predictor.txt


In [2]:
def preprocess_text(filename):
    with open(filename, 'r', encoding='utf-8') as file:
        data = file.read().lower()

    # Remove punctuation
    clean_data = data.translate(str.maketrans('', '', string.punctuation))
    corpus = clean_data.split('\n')
    # Filter out empty lines
    corpus = [line.strip() for line in corpus if line.strip()]
    return corpus

corpus = preprocess_text(file_name)
print(f"Total lines in corpus: {len(corpus)}")

Total lines in corpus: 1547


In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

X = input_sequences[:, :-1]
Y = input_sequences[:, -1]
Y = tf.keras.utils.to_categorical(Y, num_classes=total_words)

print(f"Total sequences generated: {len(input_sequences)}")

Total sequences generated: 26081


In [7]:
# Updated Cell 3: Architecture for 1,547 lines
from tensorflow.keras.optimizers import Adam

tf.keras.backend.clear_session()

model = Sequential([
    # Increase embedding to 256 to capture more nuance in your 1.5k lines
    Embedding(total_words, 256, input_length=max_sequence_len-1),
    # Use a larger LSTM layer to hold more "memory"
    LSTM(512, return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dropout(0.3),
    Dense(total_words, activation='softmax')
])

# Use a slightly higher learning rate to "jump-start" the learning
optimizer = Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Updated Cell 4: Training
# Using a smaller batch size (32) forces the model to update weights more often
history = model.fit(X, Y, epochs=100, batch_size=32, verbose=1)

Epoch 1/100
 27/816 ━━━━━━━━━━━━━━━━━━━━ 1:46:53 8s/step - accuracy: 0.0200 - loss: 8.2226

KeyboardInterrupt: 

In [8]:
def generate_text(seed_text, next_words, model, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Use a generic starting phrase to test
print(generate_text("the", 5, model, max_sequence_len))

the the the the the the
